In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import io, stats
import scipy.io as sio
import h5py
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [8]:
rchivo_mat = 'water_dataset.mat' # Asegúrate de que sea el nombre exacto de tu archivo

try:
    # Intento 1: Como archivo MATLAB v4/v5 estándar
    data = sio.loadmat(archivo_mat)
    print("¡Cargado exitosamente con SciPy (v4/v5)!")
    print("Variables disponibles:", [k for k in data.keys() if not k.startswith('__')])
    
    Y_tr = data['Y_tr']
    print(f"Forma de Y_tr: {Y_tr.shape}")

except Exception as e_scipy:
    print("SciPy falló, intentando con HDF5 (MATLAB v7.3)...")
    try:
        # Intento 2: Como archivo HDF5 (MATLAB v7.3)
        with h5py.File(archivo_mat, 'r') as f:
            print("¡Cargado exitosamente con HDF5 (v7.3)!")
            print("Variables disponibles:", list(f.keys()))
            
            # En HDF5 las matrices de celdas se leen diferente, por ejemplo:
            Y_tr = np.array(f['Y_tr'])
            print(f"Forma de Y_tr: {Y_tr.shape}")
            
    except Exception as e_hdf5:
        print("\n[ERROR CRÍTICO]: El archivo no pudo ser leído ni por SciPy ni por HDF5.")
        print("Esto suele pasar si el archivo descargado de UCI es un archivo ZIP comprimido o una página HTML.")
        print("Descomprime el archivo .zip descargado y asegúrate de apuntar al archivo .mat interno.")


SciPy falló, intentando con HDF5 (MATLAB v7.3)...

[ERROR CRÍTICO]: El archivo no pudo ser leído ni por SciPy ni por HDF5.
Esto suele pasar si el archivo descargado de UCI es un archivo ZIP comprimido o una página HTML.
Descomprime el archivo .zip descargado y asegúrate de apuntar al archivo .mat interno.


In [9]:
import pandas as pd
import numpy as np

# 1. Asegurarnos de extraer todas las variables del diccionario cargado con SciPy
features = data['features']
location_ids = data['location_ids']
X_tr = data['X_tr']
Y_tr = data['Y_tr']

# 2. Extraer las dimensiones del dataset de entrenamiento
num_locaciones, num_dias = Y_tr.shape  # 37 y 423
num_features = features.shape[1]      # Debería ser 11

# 3. Inicializar listas para reconstruir la estructura tabular
filas_datos = []

# 4. Recorrer cada día y cada estación para aplanar la estructura espacio-temporal
for dia in range(num_dias):
    # Extraer la matriz de características del día actual: forma (37, 11)
    matriz_dia = X_tr[0, dia] 
    
    for loc in range(num_locaciones):
        loc_id = location_ids[loc, 0]
        y_valor = Y_tr[loc, dia] # El pH real a predecir para esta estación en este día
        
        # Extraer las 11 características de esta estación en este día específico
        feats_estacion = matriz_dia[loc, :]
        
        # Crear un diccionario con la fila tabular
        fila = {
            'location_id': loc_id,
            'day_index': dia,
            'target_ph': y_valor
        }
        
        # Añadir dinámicamente cada característica usando los nombres del arreglo 'features'
        for f_idx, f_name in enumerate(features[0]):
            # Limpiamos el nombre de la característica por si viene en formato array o con espacios
            nombre_limpio = str(f_name[0]).strip() if isinstance(f_name[0], np.bytes_) else str(f_name).strip()
            fila[nombre_limpio] = feats_estacion[f_idx]
            
        filas_datos.append(fila)

# 5. Convertir a DataFrame de Pandas
df_agua = pd.DataFrame(filas_datos)

print("¡Aplanamiento completado con éxito!")
print(f"Dimensiones del DataFrame tabular resultante: {df_agua.shape}")
print(df_agua.head())


NameError: name 'data' is not defined

In [ ]:
df_agua.columns

Index(['location_id', 'day_index', 'target_ph',
       '['Specific conductance, water, unfiltered, microsiemens per centimeter at 25 degrees Celsius (Maximum)']',
       '['pH, water, unfiltered, field, standard units (Maximum)']',
       '['pH, water, unfiltered, field, standard units (Minimum)']',
       '['Specific conductance, water, unfiltered, microsiemens per centimeter at 25 degrees Celsius (Minimum)']',
       '['Specific conductance, water, unfiltered, microsiemens per centimeter at 25 degrees Celsius (Mean)']',
       '['Dissolved oxygen, water, unfiltered, milligrams per liter (Maximum)']',
       '['Dissolved oxygen, water, unfiltered, milligrams per liter (Mean)']',
       '['Dissolved oxygen, water, unfiltered, milligrams per liter (Minimum)']',
       '['Temperature, water, degrees Celsius (Mean)']',
       '['Temperature, water, degrees Celsius (Minimum)']',
       '['Temperature, water, degrees Celsius (Maximum)']'],
      dtype='str')

In [ ]:
# Limpiar las columnas eliminando corchetes, comillas simples y espacios extra
df_agua.columns = [col.strip("[]'") for col in df_agua.columns]

# Verificamos las nuevas columnas limpias
print(df_agua.columns)


Index(['location_id', 'day_index', 'target_ph',
       'Specific conductance, water, unfiltered, microsiemens per centimeter at 25 degrees Celsius (Maximum)',
       'pH, water, unfiltered, field, standard units (Maximum)',
       'pH, water, unfiltered, field, standard units (Minimum)',
       'Specific conductance, water, unfiltered, microsiemens per centimeter at 25 degrees Celsius (Minimum)',
       'Specific conductance, water, unfiltered, microsiemens per centimeter at 25 degrees Celsius (Mean)',
       'Dissolved oxygen, water, unfiltered, milligrams per liter (Maximum)',
       'Dissolved oxygen, water, unfiltered, milligrams per liter (Mean)',
       'Dissolved oxygen, water, unfiltered, milligrams per liter (Minimum)',
       'Temperature, water, degrees Celsius (Mean)',
       'Temperature, water, degrees Celsius (Minimum)',
       'Temperature, water, degrees Celsius (Maximum)'],
      dtype='str')


In [ ]:
# Supongamos que queremos filtrar un subconjunto de estaciones específicas
estaciones_zona_atlanta = [
    1,
    2,
    3,
    4,
    5,
]  # Reemplaza con los IDs reales de location_id
df_zona = df_agua[df_agua["location_id"].isin(estaciones_zona_atlanta)].copy()

print(f"Registros en la zona seleccionada: {len(df_zona)}")


NameError: name 'df_agua' is not defined